# 1) Autores

Matías Djukic

José María Duro Agea

# 2) Predicciones en .csv

## Descripción del Notebook

Este notebook constituye la **segunda entrega** del proyecto de predicción. Su único propósito es cargar el modelo final ya entrenado y generar el archivo de predicciones para la competición.

### Arquitectura del modelo final

El modelo cargado (`modelo_final.joblib`) es una **SVM RBF** optimizada con los siguientes componentes:

| Componente | Decisión |
|---|---|
| Escalado numérico | `RobustScaler` (robusto ante outliers detectados en el EDA) |
| Transformación de `pdays` | `PdaysTransformer` personalizado (extrae variable binaria `contactado_previamente`) |
| Codificación categórica | `OneHotEncoder` con `drop='first'` |
| Algoritmo | `SVC` con kernel RBF — hiperparámetros optimizados mediante Optuna (Sección 5.6) |
| Entrenamiento final | 100% de los datos (train + test) — Sección 6.3 |

### Preprocesamiento aplicado

- Se elimina la variable `duration` por ser un **data leakage** documentado en el EDA (Sección 3.2.5): esta variable no está disponible antes de realizar la llamada, por lo que incluirla en producción sería deshonesto.
- El resto del preprocesamiento (imputación, escalado, encoding) lo gestiona automáticamente el pipeline interno del modelo.

### Sobre la clase `PdaysTransformer`

Es obligatorio redefinirla en este notebook aunque no se use directamente. `joblib` necesita encontrar la clase en el entorno de ejecución para poder deserializar el pipeline correctamente.

In [1]:
import joblib
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

# Obligatorio redefinir la clase para que joblib pueda deserializar el pipeline
class PdaysTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mediana_contactados_ = 0
    def fit(self, X, y=None):
        mask = X['pdays'] != -1
        self.mediana_contactados_ = X.loc[mask, 'pdays'].median() if mask.sum() > 0 else 0
        return self
    def transform(self, X):
        X_trans = X.copy()
        X_trans['contactado_previamente'] = (X_trans['pdays'] != -1).astype(int)
        X_trans.loc[X_trans['pdays'] == -1, 'pdays'] = self.mediana_contactados_
        return X_trans

# 1. Cargamos el modelo ya entrenado
modelo = joblib.load('modelo_final.joblib')
print("Modelo cargado correctamente.")

# 2. Cargamos los datos de la competición
df_comp = pd.read_pickle('banks/bank_competition.pkl')
print(f"Datos de competición: {df_comp.shape}")

# 3. Misma limpieza manual que en el training (solo duration,
#    el pipeline se encarga del resto automáticamente)
if 'duration' in df_comp.columns:
    df_comp = df_comp.drop(columns=['duration'])

# 4. Generamos predicciones y las convertimos a 'yes'/'no'
predicciones = modelo.predict(df_comp)
etiquetas = ['yes' if p == 1 else 'no' for p in predicciones]

# 5. Guardamos el CSV de entrega
pd.DataFrame({
    'id': df_comp.index,
    'deposit': etiquetas
}).to_csv('predicciones.csv', index=False)

print(f"'predicciones.csv' generado con {len(etiquetas)} predicciones.")

Modelo cargado correctamente.
Datos de competición: (162, 16)
'predicciones.csv' generado con 162 predicciones.
